# GW170817 — Time-Marginalized Relative-Binning Likelihood

Extends the relative-binning (RB) likelihood [Zackay, Dai & Venumadhav 2018]
with analytical marginalization over the coalescence time `tc` and phase `φc`.

**Key idea**: In the RB framework the tc phase enters only at bin centres:
    r_j(tc) = r_j⁰ · exp(−i 2π f_j Δtc)

so all N_tc cross-correlations are computed as a single (N_tc × N_bins)
matrix multiply — far cheaper than N_tc full-grid evaluations.

**Variants implemented:**
1. `log_likelihood_rb` — standard RB (tc sampled explicitly)
2. `log_likelihood_rb_time_marg` — tc marginalized over a 3000-pt grid
3. `log_likelihood_rb_tc_phi_marg` — both tc and φc marginalized

**Data**: BayesWave-cleaned 1024-s strain, 128-s analysis segment, 4096 Hz

In [ ]:
# ── Google Colab environment setup ──────────────────────────────────────
# Run this cell first on Colab; it is a no-op on local machines.
import os, subprocess, sys

COLAB    = "google.colab" in sys.modules
REPO_DIR = "/content/mlgw_bns_jax" if COLAB else os.getcwd()

if COLAB:
    # ── JAX with CUDA 12 ──────────────────────────────────────────────
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "jax[cuda12]",
        "-f", "https://storage.googleapis.com/jax-releases/jax_cuda_releases.html",
    ])
    # ── Other Python dependencies ──────────────────────────────────────
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "gwpy", "h5py", "corner", "tqdm",
    ])

    # ── Clone the mlgw_bns_jax repo (model + waveform loader) ─────────
    if not os.path.isdir(REPO_DIR):
        subprocess.check_call([
            "git", "clone", "--branch", "time-marg-likelihood-pe",
            "--depth", "1",
            "https://github.com/saulo-albuquerque-phys/mlgw_bns_jax.git",
            REPO_DIR,
        ])

    # ── Clone SHARPy (GWNetwork / data loading) ───────────────────────
    _sharpy_repo = os.path.join(REPO_DIR, "_sharpy_repo")
    _sharpy_pkg  = os.path.join(_sharpy_repo, "sharpy")
    _sharpy_link = os.path.join(REPO_DIR, "sharpy")
    if not os.path.isdir(_sharpy_repo):
        subprocess.check_call([
            "git", "clone", "--depth", "1",
            "https://github.com/saulo-albuquerque-phys/sharpy.git",
            _sharpy_repo,
        ])
    if not os.path.exists(_sharpy_link):
        os.symlink(_sharpy_pkg, _sharpy_link)

    os.chdir(REPO_DIR)
    print(f"Working directory : {os.getcwd()}")
    print(f"SHARPy symlink    : {_sharpy_link}")
else:
    print("Not running on Colab — skipping environment setup.")


In [ ]:
from __future__ import annotations
import os, sys, time
import numpy as np
import matplotlib.pyplot as plt

os.environ.setdefault("JAX_PLATFORMS", "cpu")
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

print("JAX devices:", jax.devices())

TRIGGER_TIME     = 1187008882.43
SEGMENT_DURATION = 128.0
SAMPLING_RATE    = 4096
F_LOWER          = 23.0
F_UPPER          = 2000.0
DATA_START_GPS   = 1187008114
DATA_DURATION    = 1024

FIXED_RA  = 3.44616
FIXED_DEC = -0.408084

DATA_DIR = "gw170817_data"
OUTDIR   = "outdir_time_marg_rb"
LABEL    = "GW170817_time_marg_rb"
os.makedirs(OUTDIR, exist_ok=True)

N_TC    = 3000
TC_MIN  = -0.15
TC_MAX  =  0.15
TC_GRID = np.linspace(TC_MIN, TC_MAX, N_TC)
N_BINS  = 400

print(f"tc grid: {N_TC} points in [{TC_MIN}, {TC_MAX}] s")
print(f"RB bins: {N_BINS}")

In [ ]:
sys.path.insert(0, os.path.dirname(os.path.abspath(".")))
from jax_import_n_predict import load_predict

MODEL_PATH = "mlgw_bns_jax_model.h5"
_mlgw_predict = load_predict(MODEL_PATH)

import sharpy.GW_likelihood as _gw_mod
from sharpy.utils import McQ2Masses

def _template_mlgw_bns(params, frequency_array):
    mc, q = params[6], params[7]
    m1, m2 = McQ2Masses(mc, q)
    total_mass = m1 + m2
    chi1, chi2 = params[9], params[10]
    lambda_1, lambda_2 = params[11], params[12]
    phic = params[4]
    dist_mpc = jnp.exp(params[2])
    inclination = params[3]
    mlgw_params = jnp.array([q, lambda_1, lambda_2, chi1, chi2])
    hp, hc = _mlgw_predict(
        mlgw_params, frequency_array,
        total_mass=total_mass, distance_mpc=dist_mpc, inclination=inclination,
    )
    phase_factor = jnp.exp(-1j * phic)
    return hp * phase_factor, hc * phase_factor

_gw_mod.template = _template_mlgw_bns

from sharpy.GW_likelihood import GWNetwork, log_likelihood_det
import sharpy.PSDs
print("Model loaded — SHARPy template patched.")

In [ ]:
data_files = {
    "H1": os.path.join(DATA_DIR, f"H-H1_BWCLEANED_4KHZ-{DATA_START_GPS}-{DATA_DURATION}.txt"),
    "L1": os.path.join(DATA_DIR, f"L-L1_BWCLEANED_4KHZ-{DATA_START_GPS}-{DATA_DURATION}.txt"),
    "V1": os.path.join(DATA_DIR, f"V-V1_BWCLEANED_4KHZ-{DATA_START_GPS}-{DATA_DURATION}.txt"),
}
for det, f in data_files.items():
    assert os.path.isfile(f), f"Missing: {f}"
    print(f"{det}: {os.path.basename(f)}")

detector_settings = {}
for det in ["H1", "L1", "V1"]:
    detector_settings[det] = dict(
        data_file=data_files[det], channel="GWOSC",
        trigger_time=TRIGGER_TIME, duration=SEGMENT_DURATION,
        sampling_rate=SAMPLING_RATE, f_lower=F_LOWER, f_upper=F_UPPER,
        psd_file=None, psd_method="welch",
        download_data=False, zero_noise=False,
    )

print(f"\nBuilding GW network (segment={SEGMENT_DURATION}s)...")
t0 = time.time()
gw_network = GWNetwork(detector_settings, injection_parameters=None)
print(f"Network built in {time.time() - t0:.2f} s")
batched_detector = gw_network.batched_detector

In [ ]:
from gwpy.timeseries import TimeSeries

MERGER_GPS   = TRIGGER_TIME
T_START_PLOT = MERGER_GPS - 3.0
T_END_PLOT   = MERGER_GPS + 3.0
F_MIN, F_MAX = 20.0, 800.0
DET_COLORS = {"H1": "Reds", "L1": "Blues", "V1": "Purples"}

def _qtransform(filepath):
    strain = np.loadtxt(filepath, comments="#")
    ts = TimeSeries(strain, sample_rate=SAMPLING_RATE, t0=DATA_START_GPS)
    ts_w = ts.whiten(4, 2)
    ts_c = ts_w.crop(T_START_PLOT - 1, T_END_PLOT + 1)
    return ts_c.q_transform(frange=(F_MIN, F_MAX), qrange=(4, 64),
                             outseg=(T_START_PLOT, T_END_PLOT), logf=True)

# Raw vs BayesWave-cleaned L1
raw_l1 = os.path.join(DATA_DIR, f"L-L1_GWOSC_4KHZ_R1-{DATA_START_GPS}-{DATA_DURATION}.txt")
if os.path.isfile(raw_l1):
    qt_raw = _qtransform(raw_l1)
    qt_cln = _qtransform(data_files["L1"])
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 5), sharey=True)
    for ax, qt, title in [(ax1, qt_raw, "L1 — Raw"), (ax2, qt_cln, "L1 — BayesWave Cleaned")]:
        pcm = ax.pcolormesh(qt.times.value - MERGER_GPS, qt.frequencies.value,
                             qt.value.T, cmap="Blues", vmin=0, vmax=25)
        ax.set_yscale("log"); ax.set_ylim(F_MIN, F_MAX)
        ax.set_xlabel("Time relative to merger [s]", fontsize=12)
        ax.set_title(title, fontsize=13)
        ax.axvline(0, color="white", ls="--", lw=1)
        fig.colorbar(pcm, ax=ax).set_label("Normalized energy")
    ax1.set_ylabel("Frequency [Hz]", fontsize=12)
    fig.tight_layout()
    fig.savefig(os.path.join(OUTDIR, f"{LABEL}_L1_comparison.png"), dpi=150)
    plt.show()

# All detectors
fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharex=True)
for ax, det in zip(axes, ["H1", "L1", "V1"]):
    qt = _qtransform(data_files[det])
    pcm = ax.pcolormesh(qt.times.value - MERGER_GPS, qt.frequencies.value,
                         qt.value.T, cmap=DET_COLORS[det], vmin=0, vmax=25)
    ax.set_yscale("log"); ax.set_ylim(F_MIN, F_MAX)
    ax.set_ylabel("Frequency [Hz]", fontsize=12)
    ax.set_title(f"{det} (BayesWave cleaned)", fontsize=12)
    ax.axvline(0, color="white", ls="--", lw=1)
    fig.colorbar(pcm, ax=ax).set_label("Normalized energy")
axes[-1].set_xlabel("Time relative to merger [s]", fontsize=12)
fig.suptitle("GW170817 — Q-transform spectrograms (BayesWave cleaned)", fontsize=14)
fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, f"{LABEL}_spectrograms.png"), dpi=150)
plt.show()

## Relative-Binning Time Marginalization

In the standard RB framework the waveform ratio at bin centre j is:
    r_j = h_bins(f_j; θ) / h₀_bins(f_j)

The coalescence time tc enters only through an overall phase:
    h_bins(f_j; tc) ≈ h_bins(f_j; tc₀) · exp(−i 2π f_j (tc − tc₀))

So the ratio factorises as:
    r_j(tc) = r_j⁰ · exp(−i 2π f_j Δtc)    where Δtc = tc − tc₀

The per-detector log-likelihood at tc:
    log L_det(tc) = −TwoDTN_det · (dd_det − 2·Re[Σ_j r_j⁰·A0_j·exp(−i2πfΔtc)] + |r⁰|²B0_det)

All N_tc cross-correlations computed as:
    cross_tc = Re[ phase_matrix @ (r⁰ · A0) ]
    phase_matrix[k,j] = exp(−i 2π f_j Δtc_k)   shape (N_tc, N_bins)

Time marginalization:   log L_marg = logsumexp_k log L(tc_k) − log N_tc
Phase marginalization:  log L_φ(tc) = const + log I₀(2|W(tc)|)

In [ ]:
from functools import partial
from relative_binning import (
    build_rb_likelihood,
    build_rb_likelihood_time_marg,
    build_rb_likelihood_tc_phi_marg,
)

FIDUCIAL_PARAMS = np.array([
    FIXED_RA, FIXED_DEC,
    np.log(40.0),   # logdist: 40 Mpc
    2.545,          # theta_jn
    0.0,            # phic
    0.0,            # pol
    1.1975,         # mc
    0.87,           # q
    0.0,            # tc
    0.0,            # chi1
    0.0,            # chi2
    400.0,          # lambda1
    400.0,          # lambda2
])

# ── Standard RB ──────────────────────────────────────────────────────────
print("Building standard RB likelihood...")
t0 = time.time()
log_L_rb_std, rb_network = build_rb_likelihood(
    batched_detector, FIDUCIAL_PARAMS, _template_mlgw_bns, n_bins=N_BINS,
)
print(f"  Standard RB built in {time.time()-t0:.2f}s")

# ── Time-marginalized RB ─────────────────────────────────────────────────
print("Building time-marginalized RB likelihood...")
t0 = time.time()
log_L_rb_tm, _ = build_rb_likelihood_time_marg(
    batched_detector, FIDUCIAL_PARAMS, _template_mlgw_bns,
    TC_GRID, n_bins=N_BINS,
)
print(f"  Time-marg RB built in {time.time()-t0:.2f}s")

# ── Time+phase-marginalized RB ───────────────────────────────────────────
print("Building time+phase-marginalized RB likelihood...")
t0 = time.time()
log_L_rb_tp, _ = build_rb_likelihood_tc_phi_marg(
    batched_detector, FIDUCIAL_PARAMS, _template_mlgw_bns,
    TC_GRID, n_bins=N_BINS,
)
print(f"  Time+phase-marg RB built in {time.time()-t0:.2f}s")

# ── Full likelihood for comparison ───────────────────────────────────────
log_L_full = partial(log_likelihood_det, detector_list=batched_detector)

# ── JIT compile ──────────────────────────────────────────────────────────
log_L_rb_std_jit = jax.jit(log_L_rb_std)
log_L_rb_tm_jit  = jax.jit(log_L_rb_tm)
log_L_rb_tp_jit  = jax.jit(log_L_rb_tp)
log_L_full_jit   = jax.jit(log_L_full)

# Helper arrays without tc and phic
_p12 = FIDUCIAL_PARAMS[[0,1,2,3,4,5,6,7,9,10,11,12]]  # remove tc (idx 8)
_p11 = FIDUCIAL_PARAMS[[0,1,2,3,5,6,7,9,10,11,12]]     # remove tc+phic

# Warm-up
_ = float(log_L_rb_std_jit(FIDUCIAL_PARAMS))
_ = float(log_L_rb_tm_jit(jnp.array(_p12)))
_ = float(log_L_rb_tp_jit(jnp.array(_p11)))
_ = float(log_L_full_jit(FIDUCIAL_PARAMS))
print("All likelihoods JIT-compiled and warmed up.")

print(f"\nAt fiducial parameters:")
print(f"  Full standard:       {float(log_L_full_jit(FIDUCIAL_PARAMS)):.2f}")
print(f"  RB standard:         {float(log_L_rb_std_jit(FIDUCIAL_PARAMS)):.2f}")
print(f"  RB tc-marg:          {float(log_L_rb_tm_jit(jnp.array(_p12))):.2f}")
print(f"  RB tc+φ-marg:        {float(log_L_rb_tp_jit(jnp.array(_p11))):.2f}")

## Accuracy comparison: full likelihood vs RB approximation

We evaluate both the full and RB time-marginalized likelihoods at 50 random
parameter points and compare. The RB approximation is accurate when the
waveform ratio r_j varies by ≲ 0.1 rad in phase within each bin.

In [ ]:
from relative_binning import build_full_likelihood_time_marg

# Build full time-marginalized likelihood for comparison
print("Building full time-marginalized likelihood for comparison...")
t0 = time.time()
log_L_full_tm = build_full_likelihood_time_marg(
    batched_detector, FIDUCIAL_PARAMS, _template_mlgw_bns, TC_GRID,
)
log_L_full_tm_jit = jax.jit(log_L_full_tm)
_ = float(log_L_full_tm_jit(jnp.array(_p12)))  # warm up
print(f"  Full time-marg built in {time.time()-t0:.2f}s")

# Sample random parameter points
rng = np.random.default_rng(42)
N_COMPARE = 50

param_mins = np.array([FIXED_RA, FIXED_DEC, np.log(1), 0.0, 0.0, 0.0, 1.185, 0.5, 0.0, -0.5, -0.5, 5.0, 5.0])
param_maxs = np.array([FIXED_RA, FIXED_DEC, np.log(75), np.pi, 2*np.pi, np.pi, 1.21, 1.0, 0.0, 0.5, 0.5, 5000.0, 5000.0])
# Fix ra, dec, tc to fiducial
random_params_13 = param_mins + rng.random((N_COMPARE, 13)) * (param_maxs - param_mins)
random_params_13[:, 0] = FIXED_RA
random_params_13[:, 1] = FIXED_DEC
random_params_13[:, 8] = 0.0  # tc fixed at fiducial for 12-param extraction

random_params_12 = random_params_13[:, [0,1,2,3,4,5,6,7,9,10,11,12]]

logL_full_list = []
logL_rb_list   = []
print(f"Comparing {N_COMPARE} random parameter points...")
for i in range(N_COMPARE):
    p12 = jnp.array(random_params_12[i])
    logL_full_list.append(float(log_L_full_tm_jit(p12)))
    logL_rb_list.append(float(log_L_rb_tm_jit(p12)))

logL_full_arr = np.array(logL_full_list)
logL_rb_arr   = np.array(logL_rb_list)
delta_logL    = logL_rb_arr - logL_full_arr

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(logL_full_arr, logL_rb_arr, alpha=0.7, s=40, c="tab:blue")
lim = [min(logL_full_arr.min(), logL_rb_arr.min()),
       max(logL_full_arr.max(), logL_rb_arr.max())]
ax.plot(lim, lim, "k--", lw=1.5, label="y = x (perfect agreement)")
ax.set_xlabel("Full likelihood (time-marg)", fontsize=12)
ax.set_ylabel("RB likelihood (time-marg)", fontsize=12)
ax.set_title("RB vs Full: log L comparison", fontsize=13)
ax.legend()

ax = axes[1]
ax.hist(delta_logL, bins=20, color="tab:orange", edgecolor="black", alpha=0.8)
ax.axvline(0, color="red", ls="--", lw=1.5)
ax.set_xlabel(r"$\Delta \log L$ (RB − Full)", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_title(f"Residuals: median={np.median(delta_logL):.2f}, σ={np.std(delta_logL):.2f}", fontsize=13)

fig.suptitle(f"Accuracy of RB time-marg vs Full time-marg ({N_COMPARE} points)", fontsize=14)
fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, f"{LABEL}_accuracy_comparison.png"), dpi=150)
plt.show()

print(f"\nAccuracy statistics (ΔlogL = RB − Full):")
print(f"  Median: {np.median(delta_logL):.3f}")
print(f"  Std:    {np.std(delta_logL):.3f}")
print(f"  Max |ΔlogL|: {np.max(np.abs(delta_logL)):.3f}")

In [ ]:
# Scan over Mc and compare peaks
mc_grid = np.linspace(1.185, 1.210, 100)
logL_full_mc  = []
logL_rb_mc    = []

for mc in mc_grid:
    p12 = _p12.copy(); p12[6] = mc
    logL_full_mc.append(float(log_L_full_tm_jit(jnp.array(p12))))
    logL_rb_mc.append(float(log_L_rb_tm_jit(jnp.array(p12))))

logL_full_mc = np.array(logL_full_mc)
logL_rb_mc   = np.array(logL_rb_mc)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(mc_grid, logL_full_mc - logL_full_mc.max(), lw=2, label="Full time-marg")
ax.plot(mc_grid, logL_rb_mc   - logL_rb_mc.max(),   lw=2, ls="--", label="RB time-marg")
ax.axvline(1.1975, color="red", ls=":", lw=2, label="Paper MAP (1.1975 M☉)")
ax.axvline(mc_grid[np.argmax(logL_full_mc)], color="tab:blue", ls="-.", alpha=0.7,
           label=f"Full peak: {mc_grid[np.argmax(logL_full_mc)]:.4f} M☉")
ax.axvline(mc_grid[np.argmax(logL_rb_mc)],   color="tab:orange", ls="-.", alpha=0.7,
           label=f"RB peak:   {mc_grid[np.argmax(logL_rb_mc)]:.4f} M☉")
ax.set_xlabel(r"$\mathcal{M}_c$ [$M_\odot$]", fontsize=13)
ax.set_ylabel("log L − max(log L)", fontsize=13)
ax.set_title(r"Peak recovery: $\mathcal{M}_c$ scan", fontsize=14)
ax.legend(fontsize=10)
fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, f"{LABEL}_peak_recovery_mc.png"), dpi=150)
plt.show()

In [ ]:
import timeit
N_REPEATS = 20

t_full     = timeit.timeit(lambda: float(log_L_full_jit(FIDUCIAL_PARAMS)), number=N_REPEATS) / N_REPEATS
t_rb_std   = timeit.timeit(lambda: float(log_L_rb_std_jit(FIDUCIAL_PARAMS)), number=N_REPEATS) / N_REPEATS
t_full_tm  = timeit.timeit(lambda: float(log_L_full_tm_jit(jnp.array(_p12))), number=N_REPEATS) / N_REPEATS
t_rb_tm    = timeit.timeit(lambda: float(log_L_rb_tm_jit(jnp.array(_p12))), number=N_REPEATS) / N_REPEATS
t_rb_tp    = timeit.timeit(lambda: float(log_L_rb_tp_jit(jnp.array(_p11))), number=N_REPEATS) / N_REPEATS

print(f"Timing benchmark ({N_REPEATS} reps each):")
print(f"  Full standard:          {t_full*1e3:8.1f} ms")
print(f"  RB standard:            {t_rb_std*1e3:8.1f} ms  ({t_full/t_rb_std:.1f}× speedup)")
print(f"  Full tc-marg ({N_TC}pt):  {t_full_tm*1e3:8.1f} ms")
print(f"  RB tc-marg ({N_TC}pt):    {t_rb_tm*1e3:8.1f} ms  ({t_full_tm/t_rb_tm:.1f}× speedup vs full-marg)")
print(f"  RB tc+φ-marg ({N_TC}pt):  {t_rb_tp*1e3:8.1f} ms")

labels = ["Full\nstd", f"RB\nstd\n({N_BINS}bins)", f"Full\ntc-marg\n({N_TC}tc)",
          f"RB\ntc-marg\n({N_TC}tc)", f"RB\ntc+φ-marg\n({N_TC}tc)"]
times  = [t_full*1e3, t_rb_std*1e3, t_full_tm*1e3, t_rb_tm*1e3, t_rb_tp*1e3]
colors = ["tab:blue","tab:orange","tab:blue","tab:orange","tab:green"]
hatches = ["", "", "//", "//", "//"]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(labels, times, color=colors, alpha=0.8, edgecolor="black", hatch=None)
for bar, hatch, t in zip(bars, hatches, times):
    bar.set_hatch(hatch)
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{t:.1f}ms", ha="center", va="bottom", fontsize=10)
ax.set_ylabel("Time per call [ms]", fontsize=12)
ax.set_title("Likelihood evaluation times (full vs RB, with/without tc marginalization)", fontsize=13)
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor="tab:blue", label="Full grid"),
                   Patch(facecolor="tab:orange", label="Relative binning"),
                   Patch(facecolor="tab:green", label="tc+φ marginalized")]
ax.legend(handles=legend_elements, fontsize=11)
fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, f"{LABEL}_timing.png"), dpi=150)
plt.show()

In [ ]:
n_full = len(np.array(batched_detector.Frequency[0]))
n_bins_actual = len(np.array(rb_network.f_bins))

print("=" * 65)
print("SUMMARY — Time-Marginalized Relative Binning for GW170817")
print("=" * 65)
print(f"\nFrequency grid: {n_full} points")
print(f"RB bins:        {n_bins_actual} points ({n_full // n_bins_actual}× reduction)")
print(f"tc grid:        {N_TC} points in [{TC_MIN}, {TC_MAX}] s")
print(f"\nLog L at fiducial parameters:")
print(f"  Full standard:       {float(log_L_full_jit(FIDUCIAL_PARAMS)):.2f}")
print(f"  RB standard:         {float(log_L_rb_std_jit(FIDUCIAL_PARAMS)):.2f}")
print(f"  RB tc-marg:          {float(log_L_rb_tm_jit(jnp.array(_p12))):.2f}")
print(f"  RB tc+φ-marg:        {float(log_L_rb_tp_jit(jnp.array(_p11))):.2f}")
print(f"\nAccuracy (vs full tc-marg, {N_COMPARE} random points):")
print(f"  Median ΔlogL: {np.median(delta_logL):.3f}")
print(f"  Max |ΔlogL|:  {np.max(np.abs(delta_logL)):.3f}")
print(f"\nSpeed summary:")
print(f"  RB speedup over full grid:         {t_full/t_rb_std:.1f}×")
print(f"  RB tc-marg speedup over full-marg: {t_full_tm/t_rb_tm:.1f}×")
print(f"\nThe RB tc+φ-marg likelihood is ready for use in BlackJax NS PE.")